# 05 - Embedding 与缓存性能 (Embedding & Cache)

**目的**: 测量 sentence-transformers Embedding 管道每个环节的耗时，以及 Redis 缓存的加速效果  
**技术栈**: sentence-transformers (all-MiniLM-L6-v2, 384d) + Redis (embedding 缓存)

**链路分解**:
```
文本输入
  ↓ [A] 模型加载（冷启动，只发生一次）
  ↓ [B] Tokenization（文本 → token ids）
  ↓ [C] 推理（token ids → 384维向量）
  ↓ [D] Redis 写入（向量 → cache）
  ↓ [E] Redis 读取（cache hit）
  ↓ [F] 批量 vs 单条对比
```

## 0. 配置参数

In [ ]:
import os
from datetime import datetime

# =============================================
# 实验配置
# =============================================

# Embedding 模型（必须与生产环境一致）
MODEL_NAME = "all-MiniLM-L6-v2"  # 项目实际使用
EXPECTED_DIM = 384

# 测试文本集（模拟真实采购对话片段）
TEST_TEXTS_SHORT = [
    "服务器",
    "办公电脑",
    "采购需求",
]
TEST_TEXTS_MEDIUM = [
    "我需要采购100台服务器用于AI训练",
    "研发部门需要配置高性能工作站",
    "请推荐适合数据中心的UPS电源解决方案",
]
TEST_TEXTS_LONG = [
    "我要为研发部门采购100台Dell PowerEdge R740服务器，配置双路Intel Xeon Gold 6248R处理器，512GB内存，用于AI模型训练，预算800万元，需要在2026年3月31日前交货到北京朝阳区，要求原厂5年保修服务以及7x24小时技术支持",
    "公司需要采购一套完整的办公自动化设备，包括200台联想ThinkPad X1 Carbon笔记本电脑，100台惠普多功能打印复印一体机，以及完整的网络基础设施，总预算1500万元，需要在2026年6月底前完成所有采购和部署工作",
]

# 批量测试 batch size 列表
BATCH_SIZES = [1, 5, 10, 20, 50]

# 重复次数（单次 embedding 测试）
N_ITERATIONS = 20

# Redis 配置
REDIS_HOST = os.getenv("REDIS_HOST", "localhost")
REDIS_PORT = int(os.getenv("REDIS_PORT", "6379"))
REDIS_CACHE_PREFIX = "benchmark:embedding:"
REDIS_CACHE_TTL = 300  # 5 分钟

# AI Engine 地址（通过它调用 embedding，测量网络开销）
AI_ENGINE_URL = "http://localhost:8002"

DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)
RESULTS_FILE = f"{DATA_DIR}/embedding_cache_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"

print(f"模型: {MODEL_NAME} ({EXPECTED_DIM}d)")
print(f"测试轮数: {N_ITERATIONS}")

## 1. 依赖导入

In [ ]:
import time
import subprocess
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# 检查 sentence-transformers 是否可用
try:
    from sentence_transformers import SentenceTransformer
    ST_AVAILABLE = True
    print("sentence-transformers: 可用")
except ImportError:
    ST_AVAILABLE = False
    print("sentence-transformers: 不可用 (需在 AI Engine venv 环境运行)")
    print("提示: cd xiaocai-ai-engine && source venv/bin/activate && jupyter notebook")

print("依赖导入完成")

## 2. 模型加载延迟（冷启动 vs 热启动）

In [ ]:
if not ST_AVAILABLE:
    print("跳过：sentence-transformers 不可用")
else:
    print("=" * 55)
    print("模型加载延迟")
    print("=" * 55)

    # 冷启动（第一次加载）
    start = time.perf_counter()
    model = SentenceTransformer(MODEL_NAME)
    cold_start_ms = (time.perf_counter() - start) * 1000
    print(f"冷启动加载时间: {cold_start_ms:.0f} ms")

    # 验证模型维度
    test_vec = model.encode("test", convert_to_numpy=True)
    actual_dim = len(test_vec)
    dim_ok = actual_dim == EXPECTED_DIM
    print(f"向量维度: {actual_dim} {'(符合预期)' if dim_ok else f'(警告: 预期 {EXPECTED_DIM}!)'} ")

    # 热启动（模型已在内存中）
    hot_times = []
    for _ in range(5):
        start = time.perf_counter()
        _ = SentenceTransformer(MODEL_NAME)  # 会从缓存加载
        hot_times.append((time.perf_counter() - start) * 1000)
    print(f"热启动加载时间: {np.mean(hot_times):.0f} ms (均值)")
    print("\n注意: 生产环境中模型在服务启动时加载一次，此后为热启动")

## 3. 单文本 Embedding 延迟（按文本长度）

In [ ]:
if not ST_AVAILABLE:
    print("跳过：sentence-transformers 不可用")
else:
    print(f"单文本 Embedding 延迟测试 ({N_ITERATIONS} 轮/类型)")
    print("=" * 55)

    embed_results = []

    text_groups = [
        ("短文本(3-5字)", TEST_TEXTS_SHORT),
        ("中文本(10-20字)", TEST_TEXTS_MEDIUM),
        ("长文本(80-150字)", TEST_TEXTS_LONG),
    ]

    for group_label, texts in text_groups:
        for i in range(N_ITERATIONS):
            text = texts[i % len(texts)]
            token_count = len(text.split())

            start = time.perf_counter()
            vec = model.encode(text, convert_to_numpy=True)
            elapsed = (time.perf_counter() - start) * 1000

            embed_results.append({
                "iteration": i + 1,
                "group": group_label,
                "text_len": len(text),
                "latency_ms": elapsed,
                "vector_dim": len(vec),
            })

    df_embed = pd.DataFrame(embed_results)

    print("\nEmbedding 延迟统计 (ms):")
    embed_stats = df_embed.groupby("group")["latency_ms"].agg(
        P50=lambda x: np.percentile(x, 50),
        P95=lambda x: np.percentile(x, 95),
        Max="max",
        Mean="mean"
    ).round(2)
    print(embed_stats.to_string())

## 4. 批量 Embedding 吞吐量（Batch Size 对比）

In [ ]:
if not ST_AVAILABLE:
    print("跳过：sentence-transformers 不可用")
else:
    print(f"Batch Embedding 吞吐量测试")
    print("=" * 55)

    # 生成测试文本集
    base_texts = TEST_TEXTS_MEDIUM * 20  # 重复构造足够多的文本

    batch_results = []
    for batch_size in BATCH_SIZES:
        texts = base_texts[:batch_size]

        times = []
        for _ in range(5):  # 每个 batch_size 测 5 轮
            start = time.perf_counter()
            vecs = model.encode(texts, batch_size=batch_size, convert_to_numpy=True)
            elapsed = (time.perf_counter() - start) * 1000
            times.append(elapsed)

        avg_total = np.mean(times)
        avg_per_text = avg_total / batch_size
        throughput = 1000 / avg_per_text  # texts/sec

        batch_results.append({
            "batch_size": batch_size,
            "total_ms_mean": round(avg_total, 1),
            "per_text_ms": round(avg_per_text, 2),
            "throughput_texts_per_sec": round(throughput, 1),
        })
        print(f"  batch_size={batch_size:3d}: 总计 {avg_total:.1f}ms | 每条 {avg_per_text:.2f}ms | 吞吐 {throughput:.1f} texts/s")

    df_batch = pd.DataFrame(batch_results)
    print("\n结论: batch_size 越大，每条文本的边际成本越低（GPU 效果更显著）")

## 5. Redis Cache：命中 vs 未命中对比

In [ ]:
def redis_cmd(cmd: str) -> dict:
    """通过 docker exec 执行 Redis 命令"""
    full_cmd = f"docker exec xiaocai-redis redis-cli {cmd}"
    start = time.perf_counter()
    try:
        result = subprocess.run(full_cmd, shell=True, capture_output=True, text=True, timeout=3)
        elapsed = (time.perf_counter() - start) * 1000
        return {"success": result.returncode == 0, "latency_ms": elapsed, "output": result.stdout.strip()}
    except Exception as e:
        return {"success": False, "latency_ms": (time.perf_counter() - start) * 1000, "output": str(e)}


print(f"Redis Cache 命中/未命中延迟对比 ({N_ITERATIONS} 轮)")
print("=" * 55)

# 模拟 Embedding Service 的 cache 逻辑
def get_cache_key(text: str) -> str:
    return REDIS_CACHE_PREFIX + hashlib.md5(text.encode()).hexdigest()

# 模拟向量（384维）
mock_vector = [0.1] * EXPECTED_DIM
mock_vector_json = json.dumps(mock_vector)

cache_results = []
test_text = TEST_TEXTS_MEDIUM[0]
cache_key = get_cache_key(test_text)

for i in range(N_ITERATIONS):
    # 场景 A: Cache MISS（先删除 key）
    redis_cmd(f"DEL {cache_key}")

    # 模拟缓存查询（MISS 路径）
    r_miss_get = redis_cmd(f"GET {cache_key}")
    # 如果是 ST_AVAILABLE，这里应该调用真实模型
    # 简化：只测 Redis 操作延迟
    r_miss_set = redis_cmd(f"SETEX {cache_key} {REDIS_CACHE_TTL} '{mock_vector_json}'")

    total_miss_ms = r_miss_get["latency_ms"] + r_miss_set["latency_ms"]
    cache_results.append({"iteration": i+1, "scenario": "cache_miss_redis_ops", "latency_ms": total_miss_ms, "success": r_miss_get["success"]})

    # 场景 B: Cache HIT（key 已存在）
    r_hit = redis_cmd(f"GET {cache_key}")
    cache_results.append({"iteration": i+1, "scenario": "cache_hit", "latency_ms": r_hit["latency_ms"], "success": r_hit["success"]})

df_cache = pd.DataFrame(cache_results)
df_cache_ok = df_cache[df_cache["success"]]

if not df_cache_ok.empty:
    cache_stats = df_cache_ok.groupby("scenario")["latency_ms"].agg(
        P50=lambda x: np.percentile(x, 50),
        P95=lambda x: np.percentile(x, 95),
        Max="max",
        Mean="mean"
    ).round(2)
    print("\nRedis Cache 操作延迟 (ms):")
    print(cache_stats.to_string())

    if ST_AVAILABLE:
        # 实际 embedding 时间 + cache miss 开销
        sample_embed_ms = embed_stats.loc["中文本(10-20字)", "P50"] if 'embed_stats' in dir() else 50.0
        cache_hit_p50 = cache_stats.loc["cache_hit", "P50"]
        print(f"\n加速比估算:")
        print(f"  Cache MISS: ~{sample_embed_ms:.1f}ms (embedding) + {cache_stats.loc['cache_miss_redis_ops','P50']:.1f}ms (Redis)")
        print(f"  Cache HIT:  ~{cache_hit_p50:.1f}ms (只有 Redis GET)")
        if sample_embed_ms > 0:
            speedup = (sample_embed_ms + cache_stats.loc["cache_miss_redis_ops", "P50"]) / cache_hit_p50
            print(f"  加速比: ~{speedup:.1f}x")
else:
    print("Redis 不可达，跳过 Cache 测试")

## 6. 可视化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Embedding 与缓存性能分析", fontsize=14)

# 图1: 按文本长度的 Embedding 延迟
ax1 = axes[0]
if ST_AVAILABLE and 'df_embed' in dir():
    groups = df_embed["group"].unique()
    data = [df_embed[df_embed["group"] == g]["latency_ms"].values for g in groups]
    ax1.boxplot(data, labels=[g.split("(")[0] for g in groups], patch_artist=True)
    ax1.set_ylabel("延迟 (ms)")
    ax1.set_title("Embedding 延迟 × 文本长度")
else:
    ax1.text(0.5, 0.5, "需要\nsentence-transformers", ha='center', va='center', transform=ax1.transAxes)
    ax1.set_title("Embedding 延迟 × 文本长度")

# 图2: Batch Size vs 每条文本延迟
ax2 = axes[1]
if ST_AVAILABLE and 'df_batch' in dir():
    ax2.plot(df_batch["batch_size"], df_batch["per_text_ms"], "bo-", linewidth=2)
    ax2.set_xlabel("Batch Size")
    ax2.set_ylabel("每条文本延迟 (ms)")
    ax2.set_title("Batch Size 效率")
    ax2.grid(True, alpha=0.3)
    for _, row in df_batch.iterrows():
        ax2.annotate(f"{row['per_text_ms']:.1f}ms", (row["batch_size"], row["per_text_ms"]),
                     textcoords="offset points", xytext=(0, 8), ha='center', fontsize=8)
else:
    ax2.text(0.5, 0.5, "需要\nsentence-transformers", ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title("Batch Size 效率")

# 图3: Cache Hit vs Miss
ax3 = axes[2]
if 'df_cache' in dir() and not df_cache_ok.empty:
    scenarios = df_cache_ok["scenario"].unique()
    data3 = [df_cache_ok[df_cache_ok["scenario"] == s]["latency_ms"].values for s in scenarios]
    bp = ax3.boxplot(data3, labels=["Cache MISS\n(GET+SET)", "Cache HIT\n(GET)"], patch_artist=True)
    bp["boxes"][0].set_facecolor("coral")
    if len(bp["boxes"]) > 1:
        bp["boxes"][1].set_facecolor("steelblue")
    ax3.set_ylabel("延迟 (ms)")
    ax3.set_title("Redis Cache 命中率影响")
else:
    ax3.text(0.5, 0.5, "Redis 不可达", ha='center', va='center', transform=ax3.transAxes)
    ax3.set_title("Redis Cache 命中率影响")

plt.tight_layout()
chart_file = f"../data/embedding_cache_chart_{datetime.now().strftime('%Y%m%d_%H%M')}.png"
plt.savefig(chart_file, dpi=150, bbox_inches="tight")
plt.show()
print(f"图表已保存: {chart_file}")

## 7. 保存结果

In [ ]:
all_dfs = []
if ST_AVAILABLE and 'df_embed' in dir():
    df_embed["component"] = "embedding"
    all_dfs.append(df_embed[["iteration", "group", "latency_ms", "component"]])
if 'df_cache' in dir() and not df_cache.empty:
    df_cache["component"] = "redis_cache"
    all_dfs.append(df_cache[["iteration", "scenario", "latency_ms", "component"]].rename(columns={"scenario": "group"}))

if all_dfs:
    pd.concat(all_dfs, ignore_index=True).to_csv(RESULTS_FILE, index=False)
    print(f"结果已保存: {RESULTS_FILE}")
else:
    print("无结果保存（需要 sentence-transformers 或 Redis）")